## Import Libraries

In [3]:
import os
import time
import requests
from pathlib import Path

import pandas as pd

## Define Output Directory

In [7]:
WEATHER_OUTPUT_DIR = "weatherEDA_outputs"
os.makedirs(WEATHER_OUTPUT_DIR, exist_ok=True)

## Define Study Period

---

The weather period should match the electricity dataset period.

In [5]:
warmup_start_date = pd.Timestamp("2015-07-01")
test_end_date = pd.Timestamp("2026-05-07")  # exclusive end date

weather_start_date = warmup_start_date.strftime("%Y-%m-%d")
weather_end_date = (test_end_date - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

print("Weather start date:", weather_start_date)
print("Weather end date:", weather_end_date)

Weather start date: 2015-07-01
Weather end date: 2026-05-06


## Define Region-to-City Weather Mapping

---

Each EIA region is represented by one highly populated major city as a practical weather proxy.

In [6]:
region_weather_map = {
    "CAL": {
        "region_name": "California",
        "weather_city": "Los Angeles",
        "state": "California",
        "latitude": 34.0522,
        "longitude": -118.2437
    },
    "CAR": {
        "region_name": "Carolinas",
        "weather_city": "Charlotte",
        "state": "North Carolina",
        "latitude": 35.2271,
        "longitude": -80.8431
    },
    "CENT": {
        "region_name": "Central",
        "weather_city": "Kansas City",
        "state": "Missouri",
        "latitude": 39.0997,
        "longitude": -94.5786
    },
    "FLA": {
        "region_name": "Florida",
        "weather_city": "Jacksonville",
        "state": "Florida",
        "latitude": 30.3322,
        "longitude": -81.6557
    },
    "MIDA": {
        "region_name": "Mid-Atlantic",
        "weather_city": "Philadelphia",
        "state": "Pennsylvania",
        "latitude": 39.9526,
        "longitude": -75.1652
    },
    "MIDW": {
        "region_name": "Midwest",
        "weather_city": "Chicago",
        "state": "Illinois",
        "latitude": 41.8781,
        "longitude": -87.6298
    },
    "NE": {
        "region_name": "New England",
        "weather_city": "Boston",
        "state": "Massachusetts",
        "latitude": 42.3601,
        "longitude": -71.0589
    },
    "NW": {
        "region_name": "Northwest",
        "weather_city": "Seattle",
        "state": "Washington",
        "latitude": 47.6062,
        "longitude": -122.3321
    },
    "NY": {
        "region_name": "New York",
        "weather_city": "New York City",
        "state": "New York",
        "latitude": 40.7128,
        "longitude": -74.0060
    },
    "SE": {
        "region_name": "Southeast",
        "weather_city": "Atlanta",
        "state": "Georgia",
        "latitude": 33.7490,
        "longitude": -84.3880
    },
    "SW": {
        "region_name": "Southwest",
        "weather_city": "Phoenix",
        "state": "Arizona",
        "latitude": 33.4484,
        "longitude": -112.0740
    },
    "TEN": {
        "region_name": "Tennessee",
        "weather_city": "Nashville",
        "state": "Tennessee",
        "latitude": 36.1627,
        "longitude": -86.7816
    },
    "TEX": {
        "region_name": "Texas",
        "weather_city": "Houston",
        "state": "Texas",
        "latitude": 29.7604,
        "longitude": -95.3698
    }
}

## Save Region Weather Mapping

In [8]:
df_region_weather_map = (
    pd.DataFrame.from_dict(region_weather_map, orient="index")
    .reset_index()
    .rename(columns={"index": "Region"})
)

df_region_weather_map.to_csv(
    f"{WEATHER_OUTPUT_DIR}/region_weather_city_mapping.csv",
    index=False
)

df_region_weather_map

,Region,region_name,weather_city,state,latitude,longitude
0,CAL,California,Los Angeles,California,34.0522,-118.2437
1,CAR,Carolinas,Charlotte,North Carolina,35.2271,-80.8431
2,CENT,Central,Kansas City,Missouri,39.0997,-94.5786
3,FLA,Florida,Jacksonville,Florida,30.3322,-81.6557
4,MIDA,Mid-Atlantic,Philadelphia,Pennsylvania,39.9526,-75.1652
5,MIDW,Midwest,Chicago,Illinois,41.8781,-87.6298
6,NE,New England,Boston,Massachusetts,42.3601,-71.0589
7,NW,Northwest,Seattle,Washington,47.6062,-122.3321
8,NY,New York,New York City,New York,40.7128,-74.0060
9,SE,Southeast,Atlanta,Georgia,33.7490,-84.3880


## Define Historical Weather API Function

In [11]:
import time
import requests
import pandas as pd
from pathlib import Path

def fetch_open_meteo_historical_weather(
    region,
    region_name,
    city,
    state,
    latitude,
    longitude,
    start_date,
    end_date,
    max_retries=5,
    base_sleep=30
):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m",
            "precipitation"
        ]),
        "timezone": "UTC"
    }

    for attempt in range(1, max_retries + 1):
        response = requests.get(url, params=params, timeout=120)

        if response.status_code == 200:
            hourly_data = response.json()["hourly"]
            df = pd.DataFrame(hourly_data)

            df["timestamp_utc"] = pd.to_datetime(df["time"], utc=True)
            df["Region"] = region
            df["region_name"] = region_name
            df["weather_city"] = city
            df["weather_state"] = state
            df["latitude"] = latitude
            df["longitude"] = longitude

            df = df.drop(columns=["time"])

            return df

        if response.status_code == 429:
            sleep_seconds = base_sleep * attempt
            print(
                f"Rate limit for {region} ({city}). "
                f"Attempt {attempt}/{max_retries}. "
                f"Sleeping {sleep_seconds} seconds..."
            )
            time.sleep(sleep_seconds)
            continue

        print("Request failed.")
        print("Status code:", response.status_code)
        print("Response:", response.text[:500])
        response.raise_for_status()

    raise RuntimeError(f"Failed to fetch weather for {region} after {max_retries} retries.")

## Fetch Historical Weather for All Regions

In [12]:
weather_region_dir = Path(WEATHER_OUTPUT_DIR) / "by_region"
weather_region_dir.mkdir(parents=True, exist_ok=True)

weather_dfs = []

for region, info in region_weather_map.items():
    region_file = weather_region_dir / f"open_meteo_{region}_{weather_start_date}_{weather_end_date}.csv"

    if region_file.exists():
        print(f"Loading cached weather for {region} - {info['weather_city']}")
        weather_region_df = pd.read_csv(region_file)
        weather_region_df["timestamp_utc"] = pd.to_datetime(
            weather_region_df["timestamp_utc"],
            utc=True
        )
    else:
        print(f"Fetching weather for {region} - {info['weather_city']}, {info['state']}")

        weather_region_df = fetch_open_meteo_historical_weather(
            region=region,
            region_name=info["region_name"],
            city=info["weather_city"],
            state=info["state"],
            latitude=info["latitude"],
            longitude=info["longitude"],
            start_date=weather_start_date,
            end_date=weather_end_date,
            max_retries=5,
            base_sleep=30
        )

        weather_region_df.to_csv(region_file, index=False)
        print(f"Saved: {region_file}")

        # Longer pause to avoid rate limit
        time.sleep(20)

    weather_dfs.append(weather_region_df)

df_weather = pd.concat(weather_dfs, ignore_index=True)

df_weather = df_weather.sort_values(
    ["Region", "timestamp_utc"]
).reset_index(drop=True)

print("Weather dataset shape:", df_weather.shape)
print("Weather time range:", df_weather["timestamp_utc"].min(), "to", df_weather["timestamp_utc"].max())
print("Regions:", sorted(df_weather["Region"].unique()))

df_weather.head()

Fetching weather for CAL - Los Angeles, California
Saved: weatherEDA_outputs\by_region\open_meteo_CAL_2015-07-01_2026-05-06.csv
Fetching weather for CAR - Charlotte, North Carolina
Saved: weatherEDA_outputs\by_region\open_meteo_CAR_2015-07-01_2026-05-06.csv
Fetching weather for CENT - Kansas City, Missouri
Saved: weatherEDA_outputs\by_region\open_meteo_CENT_2015-07-01_2026-05-06.csv
Fetching weather for FLA - Jacksonville, Florida
Saved: weatherEDA_outputs\by_region\open_meteo_FLA_2015-07-01_2026-05-06.csv
Fetching weather for MIDA - Philadelphia, Pennsylvania
Saved: weatherEDA_outputs\by_region\open_meteo_MIDA_2015-07-01_2026-05-06.csv
Fetching weather for MIDW - Chicago, Illinois
Saved: weatherEDA_outputs\by_region\open_meteo_MIDW_2015-07-01_2026-05-06.csv
Fetching weather for NE - Boston, Massachusetts
Saved: weatherEDA_outputs\by_region\open_meteo_NE_2015-07-01_2026-05-06.csv
Fetching weather for NW - Seattle, Washington
Saved: weatherEDA_outputs\by_region\open_meteo_NW_2015-07-01_

,temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,timestamp_utc,Region,region_name,weather_city,weather_state,latitude,longitude
0,31.4,31,1.8,0.0,2015-07-01 00:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
1,29.9,35,12.7,0.0,2015-07-01 01:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
2,28.3,39,8.6,0.0,2015-07-01 02:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
3,27.9,39,1.8,0.0,2015-07-01 03:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
4,27.3,41,2.2,0.0,2015-07-01 04:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437


## Check Weather Data Quality

In [13]:
weather_missing_summary = df_weather.isna().sum().reset_index()
weather_missing_summary.columns = ["column", "missing_count"]
weather_missing_summary["missing_pct"] = (
    weather_missing_summary["missing_count"] / len(df_weather) * 100
)

weather_missing_summary

,column,missing_count,missing_pct
0,temperature_2m,0,0.0
1,relative_humidity_2m,0,0.0
2,wind_speed_10m,0,0.0
3,precipitation,0,0.0
4,timestamp_utc,0,0.0
5,Region,0,0.0
6,region_name,0,0.0
7,weather_city,0,0.0
8,weather_state,0,0.0
9,latitude,0,0.0


In [14]:
weather_region_summary = (
    df_weather
    .groupby("Region")
    .agg(
        region_name=("region_name", "first"),
        weather_city=("weather_city", "first"),
        weather_state=("weather_state", "first"),
        min_timestamp=("timestamp_utc", "min"),
        max_timestamp=("timestamp_utc", "max"),
        n_rows=("timestamp_utc", "size"),
        missing_temperature=("temperature_2m", lambda x: x.isna().sum()),
        missing_humidity=("relative_humidity_2m", lambda x: x.isna().sum()),
        missing_wind_speed=("wind_speed_10m", lambda x: x.isna().sum()),
        missing_precipitation=("precipitation", lambda x: x.isna().sum())
    )
    .reset_index()
)

weather_region_summary

,Region,region_name,weather_city,weather_state,min_timestamp,max_timestamp,n_rows,missing_temperature,missing_humidity,missing_wind_speed,missing_precipitation
0,CAL,California,Los Angeles,California,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
1,CAR,Carolinas,Charlotte,North Carolina,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
2,CENT,Central,Kansas City,Missouri,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
3,FLA,Florida,Jacksonville,Florida,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
4,MIDA,Mid-Atlantic,Philadelphia,Pennsylvania,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
5,MIDW,Midwest,Chicago,Illinois,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
6,NE,New England,Boston,Massachusetts,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
7,NW,Northwest,Seattle,Washington,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
8,NY,New York,New York City,New York,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0
9,SE,Southeast,Atlanta,Georgia,2015-07-01 00:00:00+00:00,2026-05-06 23:00:00+00:00,95112,0,0,0,0


## Save Historical Weather Dataset

In [15]:
weather_path = f"{WEATHER_OUTPUT_DIR}/open_meteo_historical_weather_region_hourly.csv"

df_weather.to_csv(weather_path, index=False)

print("Saved weather dataset to:")
print(weather_path)

Saved weather dataset to:
weatherEDA_outputs/open_meteo_historical_weather_region_hourly.csv
